## Hücre 1 — Google Drive Bağlantısı

Google Drive'ı bağlayarak veri setlerine ve çıktı dizinlerine erişim sağlıyoruz.

In [ ]:
# ============================================
# ORTAM TESPİTİ (Colab vs Local)
# ============================================

try:
    import google.colab
    IS_COLAB = True
    print("🌐 Google Colab ortamı tespit edildi")
except ImportError:
    IS_COLAB = False
    print("💻 Local ortam tespit edildi")

# Drive mount (sadece Colab'da)
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive bağlandı")
else:
    print("✅ Local mod - Drive mount atlandı")


## Hücre 2 — Dizin Yolları (Paths)

MVTec AD veri seti ve çıktı dizinlerinin yollarını tanımlıyoruz.

In [ ]:
# ============================================
# DİZİN YOLLARI (Path Setup)
# ============================================

import os

if IS_COLAB:
    # Google Colab yolları
    MVTEC_ROOT = "/content/drive/MyDrive/datasets/mvtec_ad"
    OUT_ROOT = "/content/drive/MyDrive/experiments/hybrid_results"
    print("📁 Colab path'leri kullanılıyor")
else:
    # Local yolları (Windows)
    MVTEC_ROOT = r"C:\Users\Engin Dalga\Documents\datasets\mvtec_ad"
    OUT_ROOT = os.path.join(os.getcwd(), "experiments", "hybrid_results")
    print("📁 Local path'ler kullanılıyor")

# Dizin oluştur
os.makedirs(OUT_ROOT, exist_ok=True)

# Dizin kontrolü
print("\n" + "="*60)
print("📂 DİZİN KONTROL")
print("="*60)

if not os.path.exists(MVTEC_ROOT):
    print("❌ HATA: MVTEC_ROOT dizini bulunamadı!")
    print(f"   Aranan yol: {MVTEC_ROOT}")
    print("\n📋 Yapmanız gerekenler:")
    if IS_COLAB:
        print("   1. MVTec AD veri setini Google Drive'a yükleyin")
        print("   2. Yukarıdaki MVTEC_ROOT yolunu kontrol edin")
    else:
        print("   1. MVTec AD veri setini indirin")
        print("   2. Yukarıdaki MVTEC_ROOT yolunu düzeltin")
    print("   3. Bu hücreyi yeniden çalıştırın")
else:
    print("✅ MVTEC_ROOT bulundu!")
    print(f"   📁 {MVTEC_ROOT}")
    
    # Kategori sayısını göster
    try:
        cats = [d for d in os.listdir(MVTEC_ROOT) if os.path.isdir(os.path.join(MVTEC_ROOT, d))]
        print(f"   📊 {len(cats)} kategori tespit edildi: {', '.join(cats[:5])}...")
    except:
        pass

print(f"\n✅ OUT_ROOT (Sonuçlar):")
print(f"   📁 {OUT_ROOT}")
if IS_COLAB:
    print(f"   ℹ️  Google Drive'da: experiments/hybrid_results/")
else:
    print(f"   ℹ️  Proje klasöründe: experiments/hybrid_results/")
print("="*60 + "\n")


## Hücre 3 — Kütüphane Kurulumu

Gerekli tüm Python kütüphanelerini kuruyoruz:
- **pandas**: Veri analizi için
- **ftfy, regex**: Metin işleme
- **opencv-python**: Görüntü işleme
- **scikit-learn**: Makine öğrenmesi metrikleri
- **CLIP**: OpenAI'nin CLIP modeli

In [ ]:
!pip -q install ftfy regex tqdm opencv-python scikit-learn pillow
!pip -q install git+https://github.com/openai/CLIP.git

## Hücre 4 — Kütüphane İmportları ve Cihaz Ayarları

Gerekli tüm kütüphaneleri import ediyoruz ve GPU/CPU cihaz ayarlarını yapıyoruz.

In [ ]:
import glob
import json
import time

# Reproducibility (Tekrarlanabilirlik) Sabitlemesi
import random
import numpy as np
import torch
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

import gc
import numpy as np
import pandas as pd
from PIL import Image
import cv2
from tqdm import tqdm

import torch
from sklearn.metrics import roc_auc_score

# Cihaz ayarları
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Kullanılan cihaz: {device}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Belleği: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# --- CLIP MODEL ---
import clip
import torch
# Orijinal kodda kullanılan CLIP modeli: ViT-B/16
clip_model, clip_preprocess = clip.load("ViT-B/16", device=device)
clip_model.eval()
print("\u2705 CLIP Modeli yüklendi.")

# --- DINOv2 MODEL ---
import torch
import torchvision.transforms as T
# Orijinal kodda kullanılan DINOv2 modeli: ViT-B/14
dinov2 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
dinov2.to(device)
dinov2.eval()

DINO_SIZE = 518  # vitb14 için yuksek cözünürlüklü patch uretimi
dino_tf = T.Compose([
    T.Resize((DINO_SIZE, DINO_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
print("\u2705 DINOv2 Modeli yüklendi.")


## Hücre 5 — MVTec Yardımcı Fonksiyonları

MVTec AD veri setini okumak ve işlemek için gerekli yardımcı fonksiyonlar:
- Veri seti kategorilerini listeleme
- Görüntü ve maske yükleme
- Normalizasyon işlemleri
- Path (yol) yönetimi

In [ ]:
def list_mvtec_categories(root):
    """
    MVTec AD kategorilerini listele.
    
    Args:
        root: MVTec AD kök dizini
    Returns:
        Sıralanmış kategori listesi
    """
    if not os.path.exists(root):
        raise ValueError(f"Root dizini bulunamadı: {root}")
    
    cats = [d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]
    cats.sort()
    return cats

def load_image_rgb(path, resize=None):
    """
    Görüntüyü RGB formatında yükle.
    
    Args:
        path: Görüntü dosya yolu
        resize: Yeniden boyutlandırma boyutu (None ise orijinal boyut)
    Returns:
        PIL Image objesi
    """
    try:
        img = Image.open(path).convert("RGB")
        if resize is not None:
            img = img.resize((resize, resize), Image.BICUBIC)
        return img
    except Exception as e:
        print(f"⚠️ Görüntü yükleme hatası {path}: {e}")
        return None

def load_mask(path, resize=None):
    """
    Ground truth maske yükle.
    
    Args:
        path: Maske dosya yolu
        resize: Yeniden boyutlandırma boyutu
    Returns:
        Binary numpy array (0 veya 1)
    """
    try:
        m = Image.open(path).convert("L")
        if resize is not None:
            m = m.resize((resize, resize), Image.NEAREST)
        m = np.array(m, dtype=np.uint8)
        m = (m > 127).astype(np.uint8)
        return m
    except Exception as e:
        print(f"⚠️ Maske yükleme hatası {path}: {e}")
        return None

def normalize_scores(x):
    """
    Skorları [0, 1] aralığına normalize et.
    
    Args:
        x: Numpy array
    Returns:
        Normalize edilmiş array
    """
    x = np.asarray(x, dtype=np.float32)
    if x.size == 0:
        return x
    mn, mx = x.min(), x.max()
    if mx - mn < 1e-12:
        return np.zeros_like(x)
    return (x - mn) / (mx - mn)

def l2_normalize(x, axis=1, eps=1e-12):
    """
    L2 normalizasyon uygula.
    
    Args:
        x: Numpy array
        axis: Normalizasyon ekseni
        eps: Sıfıra bölmeyi önlemek için küçük değer
    Returns:
        L2 normalize edilmiş array
    """
    n = np.linalg.norm(x, axis=axis, keepdims=True)
    return x / (n + eps)

def _basename_noext(p):
    """Dosya adını uzantısız al."""
    return os.path.splitext(os.path.basename(p))[0]

def get_mvtec_paths(root, category):
    """
    MVTec AD kategorisi için dosya yollarını al.
    
    Args:
        root: MVTec AD kök dizini
        category: Kategori adı (örn: 'capsule', 'bottle')
        
    Returns:
        train_good: Eğitim görüntüleri listesi
        test_items: Test görüntüleri [(img_path, label, mask_path_or_None)]
                   label: 0=normal, 1=anomali
    """
    cat_root = os.path.join(root, category)
    if not os.path.exists(cat_root):
        raise ValueError(f"Kategori dizini bulunamadı: {cat_root}")
    
    train_good = sorted(glob.glob(os.path.join(cat_root, "train", "good", "*.*")))
    
    if len(train_good) == 0:
        print(f"⚠️ UYARI: '{category}' için eğitim görüntüsü bulunamadı!")

    test_dir = os.path.join(cat_root, "test")
    gt_dir   = os.path.join(cat_root, "ground_truth")

    test_items = []
    if not os.path.exists(test_dir):
        print(f"⚠️ UYARI: Test dizini bulunamadı: {test_dir}")
        return train_good, test_items
    
    defect_types = [d for d in os.listdir(test_dir) if os.path.isdir(os.path.join(test_dir, d))]
    defect_types.sort()

    for d in defect_types:
        img_paths = sorted(glob.glob(os.path.join(test_dir, d, "*.*")))
        if d == "good":
            for p in img_paths:
                test_items.append((p, 0, None))
            continue

        gt_subdir = os.path.join(gt_dir, d)
        mask_paths = sorted(glob.glob(os.path.join(gt_subdir, "*.*"))) if os.path.exists(gt_subdir) else []

        mask_map = {_basename_noext(m): m for m in mask_paths}
        for m in mask_paths:
            b = _basename_noext(m)
            if b.endswith("_mask"):
                mask_map[b[:-5]] = m

        if len(mask_paths) == len(img_paths) and len(img_paths) > 0:
            fallback_pairs = dict(zip([_basename_noext(p) for p in img_paths], mask_paths))
        else:
            fallback_pairs = {}

        for p in img_paths:
            b = _basename_noext(p)
            mpath = mask_map.get(b, None)
            if mpath is None:
                mpath = fallback_pairs.get(b, None)
            test_items.append((p, 1, mpath))

    return train_good, test_items

## Hücre 6 — CLIP Model Yükleme ve İşlevler

OpenAI'nin CLIP modelini kullanarak:
- Model yükleme ve inicializasyon
- Görüntü embedding çıkarma
- Anomali skoru hesaplama
- Referans feature oluşturma
- Threshold (eşik) belirleme

In [ ]:
@torch.no_grad()
def build_clip_reference(train_good_paths, max_images=50):
    """
    Normal eğitim görüntülerinden CLIP görüntü özniteliklerini (features) çıkarır.
    """
    import gc
    ref_feats = []
    paths = train_good_paths[:max_images]
    for p in tqdm(paths, desc="📸 CLIP feature çıkarılıyor", unit="img"):
        img = load_image_rgb(p, resize=None)
        if img is None: continue
        img_tensor = clip_preprocess(img).unsqueeze(0).to(device)
        feat = clip_model.encode_image(img_tensor)
        feat /= feat.norm(dim=-1, keepdim=True)
        feat_np = feat.detach().cpu().numpy().astype(np.float32)
        ref_feats.append(feat_np[0])
    if len(ref_feats) == 0: raise ValueError("CLIP referansı oluşturulamadı.")
    ref_feats = np.array(ref_feats)
    mean_feat = np.mean(ref_feats, axis=0)
    mean_feat /= np.linalg.norm(mean_feat) # Normalize back
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    gc.collect()
    return mean_feat

def estimate_clip_threshold(train_good_paths, ref_feat, max_images=50, quantile=0.93):
    """
    Eğitim verisi üzerinden CLIP anomali skorlarını hesaplar ve threshold belirler.
    """
    scores = []
    paths = train_good_paths[:max_images]
    for p in paths:
        img = load_image_rgb(p, resize=None)
        if img is None: continue
        s = clip_anomaly_score(img, ref_feat)
        scores.append(s)
    if len(scores) == 0: return 0.5
    thr = np.quantile(scores, quantile)
    return float(thr)

def clip_anomaly_score(pil_img, ref_feat):
    """
    Tek bir test görüntüsünün CLIP yardımıyla anomali skorunu hesaplar.
    """
    img_t = clip_preprocess(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        f = clip_model.encode_image(img_t)
        f /= f.norm(dim=-1, keepdim=True)
    f_np = f[0].cpu().numpy().astype(np.float32)
    cos_sim = np.dot(f_np, ref_feat)
    return 1.0 - cos_sim


## Hücre 7 — DINOv2 Model ve Anomali Haritası

Facebook'un DINOv2 modelini kullanarak:
- Patch-level feature çıkarma
- Normal görüntülerden patch bank oluşturma
- Coreset subsampling
- k-NN (k-Nearest Neighbors) indexleme
- Pixel-level anomali haritası üretme

In [ ]:
@torch.no_grad()
def dinov2_patch_features(pil_img):
    x = dino_tf(pil_img).unsqueeze(0).to(device)
    feats = dinov2.forward_features(x)
    patch_tokens = feats["x_norm_patchtokens"]
    b, n, d = patch_tokens.shape
    h_w = int(np.sqrt(n))
    return patch_tokens[0], (h_w, h_w)

def l2_normalize(x, axis=1):
    eps = 1e-8
    norm = np.linalg.norm(x, axis=axis, keepdims=True)
    return x / (norm + eps)

def normalize_scores(scores):
    s_min, s_max = scores.min(), scores.max()
    if s_max - s_min < 1e-6:
        return scores - s_min
    return (scores - s_min) / (s_max - s_min)


In [ ]:
# ============================================
# DINO PATCH BANK VE k-NN FONKSİYONLARI
# ============================================

from sklearn.neighbors import NearestNeighbors
import gc

@torch.no_grad()
def build_dino_patch_bank(train_good_paths, max_images=30, max_patches_per_image=400, seed=42):
    """
    Normal eğitim görüntülerinden DINO patch features bankası oluştur.
    
    Args:
        train_good_paths: Normal eğitim görüntü yolları
        max_images: Maksimum görüntü sayısı
        max_patches_per_image: Görüntü başına maksimum patch sayısı
    Returns:
        Patch features bankası (numpy array)
    """
    bank = []
    paths = train_good_paths[:max_images]
    
    # Reproducibility için seed sabit
    np.random.seed(seed)
    
    for p in tqdm(paths, desc="🔨 DINO patch bank", unit="img"):
        img = load_image_rgb(p, resize=None)
        if img is None:
            continue
        
        pf, (gh, gw) = dinov2_patch_features(img)
        pf = pf.detach().cpu().numpy().astype(np.float32)
        
        # Subsample patches eğer çok fazlaysa
        if len(pf) > max_patches_per_image:
            indices = np.random.choice(len(pf), max_patches_per_image, replace=False)
            pf = pf[indices]
        
        bank.append(pf)
    
    if len(bank) == 0:
        raise ValueError("DINO patch bank oluşturulamadı.")
    
    bank = np.concatenate(bank, axis=0)
    
    # L2 normalize
    bank = l2_normalize(bank, axis=1)
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    
    print(f"   ✅ Patch bank: {bank.shape[0]} patches, {bank.shape[1]} dimensions")
    return bank.astype(np.float32)

def coreset_subsample(bank, target_size=12000, seed=42):
    """
    Coreset subsampling: Patch bankını hedef boyuta indir.
    
    Args:
        bank: Patch features bankası
        target_size: Hedef boyut
        seed: Random seed
    Returns:
        Subsample edilmiş banka
    """
    if len(bank) <= target_size:
        print(f"   ℹ️  Bank zaten hedef boyuttan küçük: {len(bank)} <= {target_size}")
        return bank
    
    np.random.seed(seed)
    indices = np.random.choice(len(bank), target_size, replace=False)
    
    print(f"   ✅ Coreset: {len(bank)} → {target_size} patches")
    return bank[indices]

def build_knn_index(bank, k=5):
    """
    k-NN index oluştur.
    
    Args:
        bank: Feature bankası
        k: Komşu sayısı
    Returns:
        NearestNeighbors index
    """
    nn = NearestNeighbors(n_neighbors=k, algorithm="auto", metric="cosine")
    nn.fit(bank)
    print(f"   ✅ k-NN index hazır (k={k})")
    return nn

@torch.no_grad()
def dino_anomaly_map_knn(pil_img, nn_index, k=5, out_size=256, apply_smoothing=False):
    """
    DINO patch features + k-NN ile anomaly map oluştur.
    
    Args:
        pil_img: PIL Image
        nn_index: k-NN index
        k: Komşu sayısı
        out_size: Çıktı boyutu
        apply_smoothing: Gaussian blur uygula
    Returns:
        Anomaly map (numpy array, [0,1] normalize)
    """
    # Patch features çıkar
    pf, (gh, gw) = dinov2_patch_features(pil_img)
    pf = pf.detach().cpu().numpy().astype(np.float32)
    
    # L2 normalize
    pf = l2_normalize(pf, axis=1)
    
    # k-NN distance hesapla
    dists, _ = nn_index.kneighbors(pf, n_neighbors=k, return_distance=True)
    
    # Mean distance over k neighbors
    score = dists.mean(axis=1)
    
    # Grid'e reshape et
    amap = score.reshape(gh, gw)
    
    # Normalize [0,1]
    amap = normalize_scores(amap)
    
    # Resize to output size
    amap = cv2.resize(amap, (out_size, out_size), interpolation=cv2.INTER_LINEAR)
    
    # Optional smoothing
    if apply_smoothing:
        amap = cv2.GaussianBlur(amap, (5, 5), 0)
    
    return amap.astype(np.float32)

print("✅ DINO patch bank ve k-NN fonksiyonları yüklendi!")


## Hücre 8 — 🔧 CLS TOKEN FONKSİYONLARI (Çalıştırın!)

Bu hücre DINOv2 CLS token fonksiyonlarını yükler.
**BU HÜCREYİ MUTLAKA ÇALIŞTIRIN!**

In [ ]:
def run_category_clip_only(category, clip_ref_max=50):
    train_good, test_items = get_mvtec_paths(MVTEC_ROOT, category)
    clip_ref = build_clip_reference(train_good, max_images=clip_ref_max)

    y_img, s_img = [], []
    for (img_path, label, mask_path) in tqdm(test_items, desc=f"Test CLIP {category}"):
        img = load_image_rgb(img_path, resize=None)
        s = clip_anomaly_score(img, clip_ref)
        y_img.append(label)
        s_img.append(s)

    auroc_img = roc_auc_score(y_img, s_img) if len(np.unique(y_img)) > 1 else float("nan")
    return {"category": category, "auroc_img": auroc_img}


## Hücre 9 — Üç Farklı Yöntem İmplementasyonu

### Üç farklı anomali tespit yaklaşımı:

1. **Hybrid (Hibrit)**: CLIP (görüntü-level filtreleme) + DINOv2 (pixel-level lokalizasyon)
   - CLIP ile hızlı ön tarama
   - Flagged görüntülerde DINOv2 ile detaylı analiz
   
2. **CLIP-only**: Sadece CLIP ile görüntü-level anomali tespiti
   - Hızlı ve hafif
   - Lokalizasyon yok
   
3. **DINO-only**: Sadece DINOv2 ile pixel-level anomali tespiti
   - En detaylı analiz
   - En yüksek hesaplama maliyeti

In [ ]:
# ============================================
# DINO-ONLY (MAX AGGREGATION - Literatür Standardı)
# ============================================

def run_category_dino_only_MAX(category, dino_ref_imgs=30, knn_k=5, eval_size=256, apply_smoothing=False):
    """
    DINO-only: Literatür standardı - max(anomaly_map) for image score.
    
    Pipeline:
    1. Build patch bank from training images
    2. For each test image:
       - Extract patch features
       - Compute k-NN distances → anomaly map
       - Image score = max(anomaly_map) ← STANDARD!
    
    Args:
        category: MVTec category
        dino_ref_imgs: Number of training images for patch bank
        knn_k: k for k-NN
        eval_size: Output anomaly map size
        apply_smoothing: Apply Gaussian blur to anomaly map
    
    Returns:
        dict: Results with auroc_img, auroc_pix_e2e
    """
    train_good, test_items = get_mvtec_paths(MVTEC_ROOT, category)
    
    # Build patch bank (NO CLS bank needed!)
    patch_bank = build_dino_patch_bank(train_good, max_images=dino_ref_imgs, max_patches_per_image=400, seed=42)
    bank_cs = coreset_subsample(patch_bank, target_size=12000, seed=42)
    nn_index = build_knn_index(bank_cs, k=knn_k)
    
    y_img, s_img = [], []
    y_pix, s_pix = [], []
    masked_count = 0
    
    for (img_path, label, mask_path) in tqdm(test_items, desc=f"Test DINO-MAX {category}"):
        img = load_image_rgb(img_path, resize=None)
        
        # Generate anomaly map
        amap = dino_anomaly_map_knn(img, nn_index, k=knn_k, out_size=eval_size, apply_smoothing=apply_smoothing)
        
        # ✅ LITERATURE STANDARD: MAX AGGREGATION
        img_score = float(np.max(amap))
        
        s_img.append(img_score)
        y_img.append(label)
        
        # Pixel-level evaluation
        has_gt = (mask_path is not None and os.path.exists(mask_path))
        if not has_gt:
            continue
        
        masked_count += 1
        gt = load_mask(mask_path, resize=eval_size).reshape(-1)
        y_pix.append(gt)
        s_pix.append(amap.reshape(-1).astype(np.float32))
    
    # Compute metrics
    auroc_img = roc_auc_score(y_img, s_img) if len(np.unique(y_img)) > 1 else float("nan")
    auroc_pix_e2e = roc_auc_score(np.concatenate(y_pix, axis=0), np.concatenate(s_pix, axis=0)) if len(y_pix) > 0 else float("nan")
    
    return {
        "category": category,
        "auroc_img": auroc_img,
        "auroc_pix_e2e": auroc_pix_e2e,
        "n_masked": int(masked_count),
        "method": "dino_max"
    }

print("✅ DINO-MAX fonksiyonu yüklendi (max aggregation)")


In [ ]:
# ============================================
# HYBRID (CLIP Gating + DINO Localization)
# ============================================

def run_category_hybrid(category, dino_ref_imgs=30, knn_k=5, eval_size=256,
                            clip_ref_max=50, clip_thr_quantile=0.93, apply_smoothing=False):
    """
    Hybrid: CLIP for gating + DINO for localization.
    
    Pipeline:
    1. CLIP scores all images (fast)
    2. If CLIP score > threshold: run DINO localization
       Else: skip DINO
    
    Args:
        category: MVTec category
        dino_ref_imgs: Number of training images for DINO patch bank
        knn_k: k for k-NN
        eval_size: Anomaly map output size
        clip_ref_max: Number of training images for CLIP reference
        clip_thr_quantile: CLIP threshold quantile (0.93 = 93rd percentile)
        apply_smoothing: Apply Gaussian blur
    
    Returns:
        dict: Results with auroc_img, auroc_pix_e2e, auroc_pix_cond, flag_rate
    """
    train_good, test_items = get_mvtec_paths(MVTEC_ROOT, category)
    
    # 1. CLIP reference
    clip_ref = build_clip_reference(train_good, max_images=clip_ref_max)
    clip_thr = estimate_clip_threshold(train_good, clip_ref, max_images=clip_ref_max, quantile=clip_thr_quantile)
    
    # 2. DINO patch bank
    patch_bank = build_dino_patch_bank(train_good, max_images=dino_ref_imgs, max_patches_per_image=400, seed=42)
    bank_cs = coreset_subsample(patch_bank, target_size=12000, seed=42)
    nn_index = build_knn_index(bank_cs, k=knn_k)
    
    y_img, s_img = [], []
    y_pix_e2e, s_pix_e2e = [], []  # End-to-end pixel AUROC
    y_pix_cond, s_pix_cond = [], []  # Conditional (only flagged images)
    flag_count = 0
    masked_count = 0
    
    for (img_path, label, mask_path) in tqdm(test_items, desc=f"Test Hybrid {category}"):
        img = load_image_rgb(img_path, resize=None)
        
        # Step 1: CLIP gating
        clip_score = clip_anomaly_score(img, clip_ref)
        s_img.append(clip_score)
        y_img.append(label)
        
        # Step 2: Conditional DINO
        has_gt = (mask_path is not None and os.path.exists(mask_path))
        
        if clip_score > clip_thr:
            # CLIP flagged as suspicious → run DINO
            flag_count += 1
            amap = dino_anomaly_map_knn(img, nn_index, k=knn_k, out_size=eval_size, apply_smoothing=apply_smoothing)
        else:
            # CLIP says normal → skip DINO (set map to zeros)
            amap = np.zeros((eval_size, eval_size), dtype=np.float32)
        
        # Pixel-level metrics
        if has_gt:
            masked_count += 1
            gt = load_mask(mask_path, resize=eval_size).reshape(-1)
            
            # E2E: All images (includes CLIP gating errors)
            y_pix_e2e.append(gt)
            s_pix_e2e.append(amap.reshape(-1).astype(np.float32))
            
            # Conditional: Only flagged images (DINO localization quality)
            if clip_score > clip_thr:
                y_pix_cond.append(gt)
                s_pix_cond.append(amap.reshape(-1).astype(np.float32))
    
    # Compute metrics
    auroc_img = roc_auc_score(y_img, s_img) if len(np.unique(y_img)) > 1 else float("nan")
    auroc_pix_e2e = roc_auc_score(np.concatenate(y_pix_e2e), np.concatenate(s_pix_e2e)) if len(y_pix_e2e) > 0 else float("nan")
    auroc_pix_cond = roc_auc_score(np.concatenate(y_pix_cond), np.concatenate(s_pix_cond)) if len(y_pix_cond) > 0 else float("nan")
    flag_rate = flag_count / len(test_items) if len(test_items) > 0 else 0.0
    
    return {
        "category": category,
        "auroc_img": auroc_img,
        "auroc_pix_e2e": auroc_pix_e2e,
        "auroc_pix_cond": auroc_pix_cond,
        "flag_rate": flag_rate,
        "n_flagged": int(flag_count),
        "n_masked": int(masked_count),
        "method": "hybrid"
    }

print("✅ Hybrid (CLIP-gated DINO) fonksiyonu yüklendi (CLIP gating + DINO localization)")


## Hücre 10 — Threshold Ablation (Eşik Değeri Seçimi)

Hybrid yöntemde CLIP skorunun % kaçlık dilimini (quantile) eşik olarak alacağımızı belirlemek için minik bir ablation çalışması:
- Neden `0.93` seçildiğini doğrulamak için farklı oranlar (%90, %93, %95, %97, %99) test edilmiştir.
- Neden `0.93` kullanıldığına dair argüman oluşturmak için `mean_pix_e2e` ve `mean_img` değerleri arasındaki dengeye bakılır.

In [ ]:
import numpy as np
import pandas as pd
import os

def sweep_threshold_quantiles(quantiles=(0.90, 0.93, 0.95, 0.97, 0.99), out_dir=OUT_ROOT):
    rows = []
    for q in quantiles:
        vals_img, vals_pc, vals_pe, vals_fr = [], [], [], []
        for c in list_mvtec_categories(MVTEC_ROOT):
            r = run_category_hybrid(c, clip_thr_quantile=q)
            vals_img.append(r["auroc_img"])
            vals_pc.append(r["auroc_pix_cond"])
            vals_pe.append(r["auroc_pix_e2e"])
            vals_fr.append(r.get("flag_rate", np.nan))
        row = {
            "quantile": q,
            "mean_img": float(np.nanmean(vals_img)),
            "mean_pix_cond": float(np.nanmean(vals_pc)),
            "mean_pix_e2e": float(np.nanmean(vals_pe)),
            "mean_flag_rate": float(np.nanmean(vals_fr)),
        }
        print("q=", q, row)
        rows.append(row)

    dfq = pd.DataFrame(rows)
    dfq.to_csv(os.path.join(out_dir, "Ablation_threshold_quantiles.csv"), index=False)
    return dfq

# Not: Bu blok uzun sürebilir, tüm MVTec üzerinde threshold taraması yapar.
# dfq = sweep_threshold_quantiles()
# dfq


## Hücre 11 — Değerlendirme Runner (CSV/JSON Kayıt)

Tüm kategoriler üzerinde seçilen yöntemi çalıştırır ve sonuçları kaydeder:
- Her kategori için metrikler hesaplanır (AUROC)
- Sonuçlar CSV formatında kaydedilir
- Özet istatistikler JSON formatında kaydedilir
- Çalışma süreleri raporlanır

In [ ]:
def eval_all(method_name: str, run_fn, out_dir: str):
    os.makedirs(out_dir, exist_ok=True)

    cats = list_mvtec_categories(MVTEC_ROOT)
    rows = []
    t0 = time.time()

    for c in cats:
        t1 = time.time()
        r = run_fn(c)
        rows.append({**{"category": c, "method": method_name, "runtime_s": time.time()-t1}, **r})

        show = []
        for k in ["auroc_img","auroc_pix_cond","auroc_pix_e2e","flag_rate"]:
            if k in r and isinstance(r[k], (int,float)) and r[k]==r[k]:
                show.append(f"{k}={r[k]:.4f}")
        print(c, " | ".join(show))

    df = np.nan_to_num(pd.DataFrame(rows), nan=np.nan)
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(out_dir, f"results_{method_name}.csv"), index=False)

    summary = {"method": method_name, "total_runtime_s": float(time.time()-t0)}
    for col in ["auroc_img","auroc_pix_cond","auroc_pix_e2e","flag_rate","runtime_s"]:
        if col in df.columns:
            summary[f"mean_{col}"] = float(np.nanmean(df[col].values))

    with open(os.path.join(out_dir, f"summary_{method_name}.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print("Saved:", out_dir)
    return df, summary

In [ ]:
# ============================================
# DINO-MAX TEST (~12 dakika)
# ============================================

df_d, sum_d = eval_all(
    "dino_max",
    lambda c: run_category_dino_only_MAX(c),
    out_dir=os.path.join(OUT_ROOT, "03_dino_only_max")
)

print("\n📊 DINO-MAX Sonuçları:")
print(f"Mean Image AUROC: {sum_d['mean_auroc_img']:.4f}")
print(f"Mean Pixel AUROC (E2E): {sum_d['mean_auroc_pix_e2e']:.4f}")
print(f"Mean Runtime: {sum_d['mean_runtime_s']:.2f}s")

In [ ]:
# ============================================
# HYBRID (CLIP-gated DINO) TEST (~9 dakika)
# ============================================

df_h, sum_h = eval_all(
    "hybrid",
    lambda c: run_category_hybrid(c, clip_thr_quantile=0.93),
    out_dir=os.path.join(OUT_ROOT, "01_hybrid")
)

print("\n📊 Hybrid (CLIP-gated) Sonuçları:")
print(f"Mean Image AUROC: {sum_h['mean_auroc_img']:.4f}")
print(f"Mean Pixel AUROC (E2E): {sum_h['mean_auroc_pix_e2e']:.4f}")
print(f"Mean Pixel AUROC (Conditional): {sum_h['mean_auroc_pix_cond']:.4f}  ← DINO çalıştığında")
print(f"Mean Flag Rate: {sum_h['mean_flag_rate']:.2%}")
print(f"Mean Runtime: {sum_h['mean_runtime_s']:.2f}s")

In [ ]:
# ============================================
# CLIP-ONLY TEST (~5 dakika)
# ============================================

df_c, sum_c = eval_all(
    "clip_only",
    lambda c: run_category_clip_only(c),
    out_dir=os.path.join(OUT_ROOT, "02_clip_only")
)

print("\n\U0001f4ca CLIP-ONLY Sonuçları:")
print(f"Mean Image AUROC: {sum_c['mean_auroc_img']:.4f}")
print(f"Mean Runtime: {sum_c['mean_runtime_s']:.2f}s")


## Hücre 12 — Karşılaştırmalı Excel Raporu Oluştur

Tüm yöntemlerin sonuçlarını karşılaştıran Excel dosyası üretir:
- **Sheet 1**: Image-level AUROC karşılaştırması
- **Sheet 2**: Pixel-level end-to-end AUROC
- **Sheet 3**: Pixel-level conditional AUROC (sadece flagged)
- **Sheet 4**: Ham veriler (tüm metrikler)

Excel dosyası pivot tablolar kullanarak kolay karşılaştırma sağlar.

In [ ]:
import pandas as pd

def export_comparison(df_h, df_c, df_d, out_path):
    all_df = pd.concat([df_h, df_c, df_d], ignore_index=True)

    piv_img = all_df.pivot_table(index="category", columns="method", values="auroc_img", aggfunc="mean")

    piv_pix_e2e = all_df.pivot_table(index="category", columns="method",
                                     values="auroc_pix_e2e", aggfunc="mean") if "auroc_pix_e2e" in all_df.columns else None
    piv_pix_cond = all_df.pivot_table(index="category", columns="method",
                                      values="auroc_pix_cond", aggfunc="mean") if "auroc_pix_cond" in all_df.columns else None

    with pd.ExcelWriter(out_path) as w:
        piv_img.to_excel(w, sheet_name="image_auroc")
        if piv_pix_e2e is not None:
            piv_pix_e2e.to_excel(w, sheet_name="pixel_e2e_auroc")
        if piv_pix_cond is not None:
            piv_pix_cond.to_excel(w, sheet_name="pixel_cond_auroc")
        all_df.to_excel(w, sheet_name="raw", index=False)

    print("Saved:", out_path)
    return piv_img

piv = export_comparison(df_h, df_c, df_d, os.path.join(OUT_ROOT, "comparison.xlsx"))
piv

## Hücre 13 — Hızlı Doğrulama (Sanity Check)

Tüm yöntemlerin ortalama metriklerini ekrana yazdırarak hızlı bir doğrulama yapar:
- Hybrid yöntem için image ve pixel AUROC değerleri
- CLIP-only için image AUROC
- DINO-only için image ve pixel AUROC değerleri
- Flag rate (CLIP'in kaç görüntüyü DINOv2'ye yönlendirdiği)

In [ ]:
print("HYBRID mean img:", np.nanmean(df_h["auroc_img"]))
print("HYBRID mean pix_e2e:", np.nanmean(df_h["auroc_pix_e2e"]))
print("HYBRID mean pix_cond:", np.nanmean(df_h["auroc_pix_cond"]))
print("HYBRID mean flag_rate:", np.nanmean(df_h.get("flag_rate", np.nan)))

print("CLIP_ONLY mean img:", np.nanmean(df_c["auroc_img"]))

print("DINO_ONLY mean img:", np.nanmean(df_d["auroc_img"]))
print("DINO_ONLY mean pix_e2e:", np.nanmean(df_d["auroc_pix_e2e"]))


## Hücre 14 — Algoritmik Karmaşıklık (Big-O Notation) Analizi

Bu çalışmada önerilen **Kademeli (Cascaded) Hibrit Mimari'nin** literatüre en büyük katkısı anomali tespitindeki hesaplama maliyetini (Computational Complexity) minimize etmesidir.

Aşağıda kullanılan 3 yöntemin (sınai ortamda test aşaması (inference) için) matematiksel zaman karmaşıklığı verilmiştir:

### 1. Parametreler
- **$N$**: Test edilecek görüntü sayısı.
- **$P$**: Bir görüntüden DINOv2 tarafından çıkarılan Patch (Bölge) sayısı. (Genellikle H×W bazında, örn: 256 veya 1024 civarı)
- **$B_{patch}$**: DINOv2 için kullanılan referans (Coreset) Bankası büyüklüğü. (~10.000 element)
- **$B_{cls}$**: Resim bütünü temsil eden CLS referans bankası. (Sadece referans görüntü sayısı kadar, örn: 30 element)
- **$k$**: k-NN komşuluk sayısı.
- **$\alpha$**: CLIP tarafından anomali şüphesi ile DINOv2'ye gönderilen resimlerin (Flagged) tüm resimlere oranı. ($0 \le \alpha \le 1$)

### 2. Yöntemlerin Big-O Değerlendirmesi

#### A) CLIP-Only Modeli
- **Mekanizma**: Görüntüden tek bir $D$ boyutlu öznitelik çıkarılır ve tek bir referans vektör ile Kosinüs Mesafesi hesaplanır.
- **Karmaşıklık**: $\mathcal{O}(N \cdot D)$
- **Sonuç**: En düşük zaman maliyeti, anında (real-time) çalışır ancak lokalizasyon (piksel düzeyi) yapamaz.

#### B) DINO-Only (CLS + Patch Map) Modeli
- **Mekanizma**: Görüntüden önce $CLS$ ile resim puanı bulunur, daha sonra her resim için içindeki $P$ adet patch'in her biri $B_{patch}$ içinde $k$-NN ile aranır.
- **Karmaşıklık**: $\mathcal{O}\big(N \cdot P \cdot B_{patch} \cdot \log(k)\big)$  *(Faiss/KD-Tree kullanılsa bile $P$ ve $B$ çarpanı maliyeti çok yüksektir)*
- **Sonuç**: Lokalizasyon muazzamdır ancak sınai ortamda banttan akan her ürüne uygulanması yüketimlidir (Darboğaz/Bottleneck yaratır).

#### C) Önerilen Kademeli Hibrit (Cascaded Hybrid) Sistem
- **Mekanizma**: Tüm $N$ resim $\mathcal{O}(1)$ maliyetli CLIP'ten geçer. Elde edilen Gating (Eşik) mekanizması sonrası sadece şüpheli bulunan $N \cdot \alpha$ kadar resim için DINOv2 Patch Araması yapılır.
- **Karmaşıklık**: $\mathcal{O}\big(N\big) + \mathcal{O}\big(N \cdot \alpha \cdot P \cdot B_{patch} \cdot \log(k)\big)$
- **Matematiksel Kazanım**: Eğer $\alpha = 0.1$ (Yani banttan akan ürünlerin sadece %10'u hatalı) ise, DINOv2'nin ağır yükü **%90 oranında budanmış (pruned)** olur.

---
*> "Hibrit mimari, sistemin genel asimptotik zaman karmaşıklığını (Big-O) doğrudan aşağı çekerek gerçek zamanlı (real-time) yüksek güvenilirlikli üretime entegre edilebilir hale getirmiştir."*

In [ ]:
# ============================================
# EMPİRİK BIG-O COMPLEXITY ANALİZİ
# ============================================

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import time

def empirical_complexity_analysis(category="bottle", test_counts=[10, 20, 50, 100]):
    """
    Gerçek runtime ölçümlerinden Big-O complexity hesapla.
    
    Yöntem:
    1. Farklı N değerleri için runtime ölç
    2. Log-log plot'ta slope hesapla
    3. Slope → Complexity degree
       - slope ≈ 1.0 → O(N)
       - slope ≈ 1.5 → O(N^1.5)
       - slope ≈ 2.0 → O(N²)
    
    Args:
        category: Test kategorisi
        test_counts: Farklı N değerleri listesi
    Returns:
        results: Runtime measurements
        complexities: Calculated complexity degrees
    """
    
    print("="*70)
    print("🔬 EMPİRİK BIG-O COMPLEXITY ANALİZİ")
    print("="*70)
    
    # Get test data
    train_good, test_items = get_mvtec_paths(MVTEC_ROOT, category)
    
    print(f"\n📝 Test kategorisi: {category}")
    print(f"   Toplam test görüntüsü: {len(test_items)}")
    print(f"   Test edilecek N değerleri: {test_counts}")
    
    # Storage for results
    results = {
        'CLIP': {'n': [], 'time': []},
        'DINO': {'n': [], 'time': []},
        'Hybrid': {'n': [], 'time': []}
    }
    
    # ----- CLIP COMPLEXITY -----
    print("\n🔵 CLIP-only complexity ölçümü...")
    clip_ref = build_clip_reference(train_good, max_images=50)
    
    for n in test_counts:
        if n > len(test_items):
            print(f"   ⚠️  N={n} test görüntü sayısını aşıyor, atlandı")
            break
        subset = test_items[:n]
        
        t0 = time.time()
        for (img_path, label, mask_path) in subset:
            img = load_image_rgb(img_path, resize=None)
            _ = clip_anomaly_score(img, clip_ref)
        elapsed = time.time() - t0
        
        results['CLIP']['n'].append(n)
        results['CLIP']['time'].append(elapsed)
        print(f"   N={n:3d}: {elapsed:6.3f}s")
    
    # ----- DINO COMPLEXITY -----
    print("\n🟢 DINO-only complexity ölçümü...")
    patch_bank = build_dino_patch_bank(train_good, max_images=30, max_patches_per_image=400, seed=42)
    bank_cs = coreset_subsample(patch_bank, target_size=12000, seed=42)
    nn_index = build_knn_index(bank_cs, k=5)
    
    for n in test_counts:
        if n > len(test_items):
            break
        subset = test_items[:n]
        
        t0 = time.time()
        for (img_path, label, mask_path) in subset:
            img = load_image_rgb(img_path, resize=None)
            _ = dino_anomaly_map_knn(img, nn_index, k=5, out_size=256)
        elapsed = time.time() - t0
        
        results['DINO']['n'].append(n)
        results['DINO']['time'].append(elapsed)
        print(f"   N={n:3d}: {elapsed:6.3f}s")
    
    # ----- HYBRID COMPLEXITY -----
    print("\n🟡 Hybrid complexity ölçümü...")
    clip_thr = estimate_clip_threshold(train_good, clip_ref, max_images=50, quantile=0.93)
    
    for n in test_counts:
        if n > len(test_items):
            break
        subset = test_items[:n]
        
        t0 = time.time()
        flag_count = 0
        for (img_path, label, mask_path) in subset:
            img = load_image_rgb(img_path, resize=None)
            clip_score = clip_anomaly_score(img, clip_ref)
            if clip_score > clip_thr:
                flag_count += 1
                _ = dino_anomaly_map_knn(img, nn_index, k=5, out_size=256)
        elapsed = time.time() - t0
        alpha = flag_count / n if n > 0 else 0
        
        results['Hybrid']['n'].append(n)
        results['Hybrid']['time'].append(elapsed)
        print(f"   N={n:3d}: {elapsed:6.3f}s (α={alpha:.2f}, flagged={flag_count}/{n})")
    
    # ----- BIG-O CALCULATION -----
    print("\n" + "="*70)
    print("📈 BIG-O COMPLEXITY CALCULATION (Log-Log Regression)")
    print("="*70)
    
    complexities = {}
    
    for method, data in results.items():
        if len(data['n']) < 2:
            print(f"\n{method}: Yetersiz data")
            continue
        
        # Log-log regression
        log_n = np.log10(data['n'])
        log_t = np.log10(data['time'])
        
        # Linear fit: log(T) = c·log(N) + b
        # Slope c = complexity degree
        coeffs = np.polyfit(log_n, log_t, 1)
        slope = coeffs[0]
        intercept = coeffs[1]
        
        complexities[method] = slope
        
        # Complexity class belirleme
        if slope < 0.7:
            complexity_class = "O(1) - Constant"
        elif slope < 1.2:
            complexity_class = "O(N) - Linear"
        elif slope < 1.7:
            complexity_class = "O(N^1.5) - Superlinear"
        elif slope < 2.5:
            complexity_class = "O(N²) - Quadratic"
        else:
            complexity_class = f"O(N^{slope:.1f}) - Polynomial"
        
        print(f"\n{method}:")
        print(f"   Slope (log-log): {slope:.3f}")
        print(f"   Intercept: {intercept:.3f}")
        print(f"   Empirical Complexity: {complexity_class}")
        print(f"   Formula: T ≈ {10**intercept:.2f} · N^{slope:.2f}")
    
    # ----- VISUALIZATION -----
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    colors = {'CLIP': '#3498db', 'DINO': '#2ecc71', 'Hybrid': '#f39c12'}
    
    # Linear plot
    for method, data in results.items():
        if len(data['n']) > 0:
            ax1.plot(data['n'], data['time'], 'o-', 
                    label=method, linewidth=2, markersize=8,
                    color=colors.get(method, 'gray'))
    ax1.set_xlabel('Number of Test Images (N)', fontsize=11)
    ax1.set_ylabel('Runtime (seconds)', fontsize=11)
    ax1.set_title('Runtime vs N (Linear Scale)', fontsize=12, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)
    
    # Log-log plot (Big-O görselleştirme)
    for method, data in results.items():
        if len(data['n']) > 0:
            slope = complexities.get(method, 0)
            ax2.loglog(data['n'], data['time'], 'o-', 
                      label=f'{method} (slope={slope:.2f})', 
                      linewidth=2, markersize=8,
                      color=colors.get(method, 'gray'))
    ax2.set_xlabel('Number of Test Images (N)', fontsize=11)
    ax2.set_ylabel('Runtime (seconds)', fontsize=11)
    ax2.set_title('Runtime vs N (Log-Log Scale)', fontsize=12, fontweight='bold')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3, which='both')
    
    plt.tight_layout()
    plt.show()
    
    # ----- SUMMARY TABLE -----
    print("\n" + "="*70)
    print("📊 COMPLEXITY SUMMARY TABLE")
    print("="*70)
    
    import pandas as pd
    summary_data = []
    for method, slope in complexities.items():
        if slope < 1.2:
            complexity = "O(N)"
        elif slope < 1.7:
            complexity = "O(N^1.5)"
        elif slope < 2.5:
            complexity = "O(N²)"
        else:
            complexity = f"O(N^{slope:.1f})"
        
        avg_time = np.mean(results[method]['time'])
        summary_data.append({
            'Method': method,
            'Slope': f"{slope:.3f}",
            'Complexity': complexity,
            'Avg Time (s)': f"{avg_time:.2f}"
        })
    
    df_summary = pd.DataFrame(summary_data)
    print(df_summary.to_string(index=False))
    
    print("\n" + "="*70)
    print("✅ COMPLEXITY ANALYSIS TAMAMLANDI!")
    print("="*70)
    print("\n📝 NOTLAR:")
    print("   • Slope ~1.0 = Linear O(N)")
    print("   • CLIP: Beklenen O(N) - her görüntü sabit işlem")
    print("   • DINO: Beklenen O(N) - her görüntü patch işleme (sabit)")
    print("   • Hybrid: Beklenen O(N·α) ≈ O(N) - α=flag rate")
    print("="*70)
    
    return results, complexities

# ⚠️ NOT: Bu hücre uzun sürebilir (~3-5 dakika)
# Test etmek için önce küçük N değerleri kullanın:
# results, complexities = empirical_complexity_analysis(category="bottle", test_counts=[10, 20, 30])

# Tam analiz için:
# results, complexities = empirical_complexity_analysis(category="bottle", test_counts=[10, 20, 50, 100])

print("\n💡 Bu hücreyi çalıştırmak için yukarıdaki comment'i kaldırın.")
print("   Örnek: results, complexities = empirical_complexity_analysis()")


## Hücre 15 — Grafiksel Karşılaştırma ve Raporlama

Tüm yöntemlerin testleri bittikten sonra metrikleri hem bir özet tablo halinde sunar hem de AUROC ve Runtime yönünden grafiklere döker:
- 1. Bar Chart: Image AUROC (Tüm Modeller)
- 2. Bar Chart: Pixel AUROC (Hybrid vs DINO)
- 3. Bar Chart: Runtime / Süre Karşılaştırması

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

def print_and_plot_summary(df_hybrid, df_clip, df_dino):
    # --- 1. Özet DataFrame Oluştur ---
    runtime_clip = df_clip['runtime_s'].sum() if 'runtime_s' in df_clip.columns else 0
    runtime_dino = df_dino['runtime_s'].sum() if 'runtime_s' in df_dino.columns else 0
    runtime_hybrid = df_hybrid['runtime_s'].sum() if 'runtime_s' in df_hybrid.columns else 0
    
    # Tasarruf Hesabı (DINO'ya göre Hybrid ne kadar tasarruf sağladı)
    time_saved_sec = runtime_dino - runtime_hybrid
    time_saved_pct = (time_saved_sec / runtime_dino * 100) if runtime_dino > 0 else 0
    
    summary_data = {
        'Model': ['CLIP-Only', 'DINO-Only (CLS)', 'Hybrid (CLS)'],
        'Image AUROC': [df_clip['auroc_img'].mean(), df_dino['auroc_img'].mean(), df_hybrid['auroc_img'].mean()],
        'Pixel AUROC (e2e)': [None, df_dino['auroc_pix_e2e'].mean(), df_hybrid['auroc_pix_e2e'].mean()],
        'Pixel AUROC (DINO|CLIP-flag)': [None, df_dino['auroc_pix_cond_clip'].mean() if 'auroc_pix_cond_clip' in df_dino.columns else None, df_hybrid['auroc_pix_cond'].mean()],
        'Pixel AUROC (Cond.)': [None, None, df_hybrid['auroc_pix_cond'].mean()],
        'Runtime (s)': [runtime_clip, runtime_dino, runtime_hybrid]
    }
    df_summary = pd.DataFrame(summary_data)
    
    print("\n" + "="*70)
    print("📊 FINAL KARŞILAŞTIRMA RAPORU")
    print("="*70)
    print(df_summary.to_string(index=False))
    print(f"   ★ Hybrid Conditional Pixel AUROC (DINO çalışınca): {df_hybrid['auroc_pix_cond'].mean():.4f}")
    print("-"*70)
    print(f"⏱️ ZAMAN TASARRUFU ANALİZİ:")
    print(f"   ▶ DINOv2 Toplam Süre: {runtime_dino:.1f} saniye")
    print(f"   ▶ Hybrid Toplam Süre: {runtime_hybrid:.1f} saniye")
    print(f"   ✅ Elde Edilen Tasarruf: {time_saved_sec:.1f} saniye (%{time_saved_pct:.1f} daha hızlı)")
    print("="*70 + "\n")
    
    # --- 2. Grafikler ---
    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Image AUROC
    sns.barplot(data=df_summary, x='Model', y='Image AUROC', ax=axes[0], palette='viridis')
    axes[0].set_title('Image AUROC Karşılaştırması')
    axes[0].set_ylim([0, 1.0])
    for p in axes[0].patches:
        axes[0].annotate(f'{p.get_height():.3f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='center', xytext=(0, 6), textcoords='offset points')

    # Pixel AUROC
    df_pixel = df_summary.dropna(subset=['Pixel AUROC (e2e)'])
    sns.barplot(data=df_pixel, x='Model', y='Pixel AUROC (e2e)', ax=axes[1], palette='flare')
    axes[1].set_title('Pixel AUROC Karşılaştırması (e2e)')
    axes[1].set_ylim([0, 1.0])
    for p in axes[1].patches:
        axes[1].annotate(f'{p.get_height():.3f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='center', xytext=(0, 6), textcoords='offset points')
                     
    # Runtime
    sns.barplot(data=df_summary, x='Model', y='Runtime (s)', ax=axes[2], palette='cividis')
    axes[2].set_title(f'Toplam Çalışma Süresi (-%{time_saved_pct:.0f} Tasarruf)')
    for p in axes[2].patches:
        axes[2].annotate(f'{p.get_height():.1f}s', (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='center', xytext=(0, 6), textcoords='offset points')

    plt.tight_layout()
    plt.show()

# Görüntüleyelim:
print_and_plot_summary(df_h, df_c, df_d)
